In [5]:
import xarray as xr
import pandas as pd

import pygrib
import sqlite3

import numpy as np
from pathlib import Path

import numpy as np



In [2]:
fils = pd.read_csv("/home/joe/work/Fire/ML/New/files_grib_2982_byvar.csv")
END = 20260331
#TARGET_VAR="10u"
#TARGET_VAR="2t"
TARGET_VAR="2d"
# TARGET_VAR="tp"
files=fils.loc[fils['variable'] == TARGET_VAR,"file"].values.tolist()
files
if TARGET_VAR == "tp":
    files = ['Data/b60a5564de59c1b8fea3beb193b71d8d.grib'] #    tp 202601010 20260401900
elif TARGET_VAR == "2t":
    files = ['Data/368f88589ed7aeaa139824a4cc0121ca.grib'] #    2t 202601010 20260402900
elif TARGET_VAR == "2d":
    files = ['Data/ea350c6b841a0759715b15ff0cce68e6.grib'] #    2d 202601010 20260402900
elif TARGET_VAR == "10u":
    files = ['Data/1a48edbeb3d7b177196fb17c7d710a5c.grib'] #    10u 202601010 20260402900
elif TARGET_VAR == "10v":
    files = ['Data/77a97b67319f589579dd9d5f890eedfb.grib'] #    10v 202601010 20260402900
#files = ["Data/fc64d1f02244ef24c26fcd3f03eb0bf2.grib"]


In [3]:


# ----------------------------
# CONFIG
# ----------------------------

GRIB_FOLDER = Path("/home/joe/work/Fire/ML/Data")
DB_PATH = f"/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_{TARGET_VAR}.sqlite"
DB_LATLONS = "/home/joe/work/Fire/ML/Data/DB/era5_latlons_2982.sqlite"
VAR_MAP = {
    "2t": "T",
    "2d": "Td",
    "10u": "u",
    "10v": "v",
    "tp": "pcp"
}

# ----------------------------
# Get Lat-Lons
# ----------------------------

conn = sqlite3.connect(DB_LATLONS)
cursor = conn.cursor()
cursor.execute("SELECT point_id, lat, lon FROM lat_lon_points")

lookup = {
    (round(lat, 2), round(lon, 2)): pid
    for pid, lat, lon in cursor.fetchall()
}

print(f"Loaded {len(lookup)} points.")
print(lookup)
conn.close()


conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS daily_data (
    datetime TEXT NOT NULL,
    point_id INTEGER NOT NULL,
    variable TEXT NOT NULL,
    '0' REAL,
    '3' REAL,
    '6' REAL,
    '9' REAL,
    '12' REAL,
    '15' REAL,
    '18' REAL,
    '21' REAL,
    PRIMARY KEY (datetime, point_id, variable)
);
""")

conn.commit()


Loaded 2982 points.
{(41.1, -109.0): 0, (41.1, -108.9): 1, (41.1, -108.8): 2, (41.1, -108.7): 3, (41.1, -108.6): 4, (41.1, -108.5): 5, (41.1, -108.4): 6, (41.1, -108.3): 7, (41.1, -108.2): 8, (41.1, -108.1): 9, (41.1, -108.0): 10, (41.1, -107.9): 11, (41.1, -107.8): 12, (41.1, -107.7): 13, (41.1, -107.6): 14, (41.1, -107.5): 15, (41.1, -107.4): 16, (41.1, -107.3): 17, (41.1, -107.2): 18, (41.1, -107.1): 19, (41.1, -107.0): 20, (41.1, -106.9): 21, (41.1, -106.8): 22, (41.1, -106.7): 23, (41.1, -106.6): 24, (41.1, -106.5): 25, (41.1, -106.4): 26, (41.1, -106.3): 27, (41.1, -106.2): 28, (41.1, -106.1): 29, (41.1, -106.0): 30, (41.1, -105.9): 31, (41.1, -105.8): 32, (41.1, -105.7): 33, (41.1, -105.6): 34, (41.1, -105.5): 35, (41.1, -105.4): 36, (41.1, -105.3): 37, (41.1, -105.2): 38, (41.1, -105.1): 39, (41.1, -105.0): 40, (41.1, -104.9): 41, (41.1, -104.8): 42, (41.1, -104.7): 43, (41.1, -104.6): 44, (41.1, -104.5): 45, (41.1, -104.4): 46, (41.1, -104.3): 47, (41.1, -104.2): 48, (41.1, -1

In [4]:

# ----------------------------
# BUILD LOOKUP DICTIONARY
# ----------------------------


# ----------------------------
# PROCESS FILES
# ----------------------------

for file in files:
    print(f"\nProcessing {file}")
    total_before = conn.total_changes

    grbs = pygrib.open(f"/home/joe/work/Fire/ML/{file}")
    combined = {}
    for grb in grbs:

        if grb.shortName not in VAR_MAP:
            continue

        varname = VAR_MAP[grb.shortName]
        vd = grb.validityDate
        vt = grb.validityTime
        valid_time = str(vd)+str(vt).zfill(4)
        if file == "Data/8bbd4475b42607d4c1253e7f07ad3ee5.grib" and float(valid_time) >  202412312399:
            continue
       
        if vd < 19900101:
             continue
        
        if vd > END:
            continue
        values = grb.values
        lats, lons = grb.latlons()

        values = values.flatten()
        lats = np.round(lats.flatten(), 2)
        lons = np.round(lons.flatten(), 2)

        records = []

        for lat, lon, val in zip(lats, lons, values):

            if np.isnan(val):
                continue

            pid = lookup[(lat, lon)]

            if pid not in combined:
                combined[pid]={}
            if vd not in combined[pid]:
                combined[pid][vd]={}
            if vt not in combined[pid][vd]:
                combined[pid][vd][vt]={}


            combined[pid][vd][vt] = float(val)
        
    pids=[]
    vds=[]
    vals={}
    for pid in combined:
        for vd, vt_dict in combined[pid].items():
            pids.append(pid)
            vds.append(vd)
            for vt in vt_dict.keys():
                if vt not in vals:
                    vals[vt]=[] 
                vals[vt].append(combined[pid][vd][vt])  
    df=pd.DataFrame({"Point": pids, "Date": vds, "0": vals[0], "3": vals[300], "6": vals[600], "9": vals[900], "12": vals[1200], "15": vals[1500], "18": vals[1800], "21": vals[2100]})
    df['Variable']= varname

    records = [
    (str(row['Date']), row['Point'], row['Variable'], row['0'], row['3'], row['6'], row['9'], row['12'], row['15'], row['18'], row['21'])
    for _, row in df.iterrows()
]

# Bulk insert with conflict handling
    cursor.executemany("""
        INSERT OR IGNORE INTO daily_data (datetime, point_id, variable, '0', '3', '6', '9', '12', '15', '18', '21')
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)
    conn.commit()
    total_after = conn.total_changes
    print(f"Rows inserted: {total_after - total_before}")


conn.close()

print("\nDone.")


Processing Data/ea350c6b841a0759715b15ff0cce68e6.grib
Rows inserted: 0

Done.


## Lat-Lon to DB

In [ ]:
import pygrib
import sqlite3
import numpy as np

file = "/home/joe/work/Fire/ML/Data/3fa97d13976d7e2ae5505a4cf12dc52.grib"
dbDir = "/home/joe/work/Fire/ML/Data/DB"
conn = sqlite3.connect(f"{dbDir}/era5_latlons_2982.sqlite")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS lat_lon_points (
    point_id INTEGER PRIMARY KEY,
    lat REAL NOT NULL,
    lon REAL NOT NULL,
    UNIQUE(lat, lon)
);
""")

grbs = pygrib.open(file)
grb = grbs[1]

lats, lons = grb.latlons()

nlat, nlon = lats.shape

records = []
point_id = 0

for i in range(nlat):          # each latitude row (top to bottom)
    for j in range(nlon):      # each longitude column (left to right)
        lat = round(float(lats[i, j]), 4)
        lon = round(float(lons[i, j]), 4)

        records.append((point_id, lat, lon))
        point_id += 1

cursor.executemany("""
INSERT OR IGNORE INTO lat_lon_points (point_id, lat, lon)
VALUES (?, ?, ?)
""", records)

conn.commit()
conn.close()

print(f"Inserted {point_id} grid points starting at 0.")

Inserted 2982 grid points starting at 0.
